# Урок 10 - AI-агенты в продакшене

В этом уроке вы узнаете о **производственных шаблонах** для AI-агентов с использованием Microsoft Agent Framework (Python). Мы рассмотрим:

- **Наблюдаемость** — добавление измерения времени и логирования взаимодействий агента
- **Оценка** — использование оценочного агента для оценки качества ответов
- **Управление затратами** — стратегии оптимизации токенов и выбора модели

Сценарий — **туристический агент**, который помогает пользователям планировать поездки, с наложенным мониторингом и оценкой.


## Установка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Производственные соображения

Перевод ИИ-агентов от прототипов к производству требует тщательного внимания к трём основным аспектам:

1. **Наблюдаемость** — Необходима видимость того, что делает агент, сколько времени это занимает и какие инструменты он вызывает. Без трассировки и логирования отладка проблем в производстве практически невозможна.

2. **Оценка** — Автоматизированные проверки качества обеспечивают сохранение точности, полноты и полезности ответов агента с течением времени. Агент-оценщик может выставлять баллы ответам по заданным критериям.

3. **Управление затратами** — Использование токенов напрямую влияет на стоимость. Стратегии, такие как оптимизация подсказок, выбор модели и кэширование, помогают контролировать расходы без ущерба для качества.


## Создание наблюдаемого агента

Мы определяем инструменты для путешествий и оборачиваем вызов агента с таймингом, чтобы мы могли отслеживать задержку. В продуктивной среде вы бы интегрировали это с OpenTelemetry или подобной системой трассировки.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Шаблоны оценки

Распространённым производственным шаблоном является использование второго агента в качестве **оценщика**. Оценщик оценивает ответ основного агента по заранее определённым критериям, таким как полнота, точность и полезность.

Это позволяет:
- Автоматически контролировать качество перед отправкой ответов пользователям
- Обнаруживать регрессии при изменении подсказок или моделей
- Постоянно мониторить производительность агента со временем


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Стратегии управления затратами

Контроль затрат имеет решающее значение для производственных AI-агентов. Вот ключевые стратегии:

| Strategy | Description |
|---|---|
| **Оптимизация подсказок** | Держите системные инструкции краткими. Удаляйте избыточный контекст для сокращения входных токенов. |
| **Выбор модели** | Используйте более маленькие, дешевые модели (например, GPT-4o-mini) для простых задач, таких как классификация или извлечение, и оставляйте более крупные модели для сложного рассуждения. |
| **Кэширование** | Кэшируйте результаты инструментов и часто задаваемые запросы, чтобы избежать избыточных вызовов API. |
| **Бюджеты токенов** | Устанавливайте ограничения `max_tokens`, чтобы предотвратить неожиданно длинные ответы. |
| **Пакетная обработка** | Группируйте несколько пользовательских запросов в один вызов API, где это возможно. |

На практике хорошо работает многоуровневый подход: направляйте простые запросы к быстрой, недорогой модели и эскалируйте только сложные запросы к более мощной.


## Резюме

В этом уроке вы научились:

1. **Добавлять наблюдаемость** к взаимодействиям агента с помощью измерения времени и логирования, создавая основу для трассировки и мониторинга.
2. **Автоматически оценивать ответы агента** с помощью агента-оценщика, который ставит баллы за полноту, точность и полезность.
3. **Управлять затратами** через оптимизацию подсказок, выбор модели, кэширование и бюджеты на токены.

Эти производственные паттерны помогают гарантировать, что ваши AI-агенты надежны, измеримы и экономичны в масштабах.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
